In [1]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 6 Commit 1: Prepare Tableau Data for Dashboard 1 & Dashboard 2
# feat(task6): export mortgage overview and model evaluation datasets
#
# Coursework Requirement:
# Prepare Tableau-ready datasets for visual analytics.
# Dashboard 1 : Mortgage Application Overview
# Dashboard 2 : Machine Learning Model Performance
# ──────────────────────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import pandas as pd
import builtins

os.environ["JAVA_HOME"]="/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"]=sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA_Task6")
    .master("local[*]")
    .config("spark.driver.memory","10g")
    .config("spark.sql.shuffle.partitions","200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ------------------------------------------------------------------------------
# Project Paths
# ------------------------------------------------------------------------------

BASE="/content/drive/MyDrive/7006SCN"

PROCESSED=f"{BASE}/data/processed"

OUTPUT=f"{BASE}/outputs"

print("="*80)
print("TASK 6 : TABLEAU DATA PREPARATION")
print("="*80)

# ------------------------------------------------------------------------------
# Load Clean HMDA Dataset
# ------------------------------------------------------------------------------

df=spark.read.parquet(
    f"{PROCESSED}/hmda_clean.parquet"
)

print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

# ==============================================================================
# DASHBOARD 1
# Mortgage Application Overview
# ==============================================================================

print("\nPreparing Dashboard 1...")

# ------------------------------------------------------------------------------
# KPI Summary
# ------------------------------------------------------------------------------

total=df.count()

approved=df.filter(
    col("loan_approval")==1
).count()

rejected=df.filter(
    col("loan_approval")==0
).count()

approval_rate = builtins.round(
    approved / total * 100,
    2
)

kpi=pd.DataFrame({

    "Metric":[
        "Total Applications",
        "Approved Loans",
        "Rejected Loans",
        "Approval Rate (%)"
    ],

    "Value":[
        total,
        approved,
        rejected,
        approval_rate
    ]

})

kpi.to_csv(
    f"{OUTPUT}/t6_dashboard1_kpi.csv",
    index=False
)

# ------------------------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------------------------

missing=[]

for c in df.columns:

    missing.append({

        "Column":c,

        "Missing Values":df.filter(
            col(c).isNull()
        ).count()

    })

missing_df=pd.DataFrame(missing)

missing_df.to_csv(

    f"{OUTPUT}/t6_dashboard1_missing.csv",

    index=False

)

# ------------------------------------------------------------------------------
# Loan Approval Distribution
# ------------------------------------------------------------------------------

approval=(
    df.groupBy("loan_approval")
      .count()
      .toPandas()
)

approval.to_csv(

    f"{OUTPUT}/t6_dashboard1_approval.csv",

    index=False

)

# ------------------------------------------------------------------------------
# Loan Purpose Distribution
# ------------------------------------------------------------------------------

if "loan_purpose" in df.columns:

    purpose=(
        df.groupBy("loan_purpose")
          .count()
          .toPandas()
    )

    purpose.to_csv(

        f"{OUTPUT}/t6_dashboard1_loanpurpose.csv",

        index=False

    )

# ------------------------------------------------------------------------------
# Race Distribution
# ------------------------------------------------------------------------------

if "applicant_race_1" in df.columns:

    race=(
        df.groupBy("applicant_race_1")
          .count()
          .toPandas()
    )

    race.to_csv(

        f"{OUTPUT}/t6_dashboard1_race.csv",

        index=False

    )

# ------------------------------------------------------------------------------
# Ethnicity Distribution
# ------------------------------------------------------------------------------

if "applicant_ethnicity_1" in df.columns:

    ethnicity=(
        df.groupBy("applicant_ethnicity_1")
          .count()
          .toPandas()
    )

    ethnicity.to_csv(

        f"{OUTPUT}/t6_dashboard1_ethnicity.csv",

        index=False

    )

print("Dashboard 1 datasets exported ✓")

# ------------------------------------------------------------------------------
# Dataset Summary
# ------------------------------------------------------------------------------

dataset_summary = pd.DataFrame({

    "Metric": [

        "Total Records",
        "Total Columns",
        "Processing Framework",
        "Target Variable"

    ],

    "Value": [

        df.count(),
        len(df.columns),
        "Apache Spark",
        "loan_approval"

    ]

})

dataset_summary.to_csv(

    f"{OUTPUT}/t6_dashboard1_dataset_summary.csv",

    index=False

)

print("Dataset summary exported ✓")

# ------------------------------------------------------------------------------
# Missing Values Summary
# ------------------------------------------------------------------------------

from pyspark.sql.functions import col, sum as spark_sum, when

missing = [

    spark_sum(

        when(col(c).isNull(),1).otherwise(0)

    ).alias(c)

    for c in df.columns

]

missing_df = (

    df.select(missing)

      .toPandas()

      .T

      .reset_index()

)

missing_df.columns = [

    "Column",

    "Missing Values"

]

missing_df.to_csv(

    f"{OUTPUT}/t6_dashboard1_missing_values.csv",

    index=False

)

print("Missing values summary exported ✓")

# ------------------------------------------------------------------------------
# Pipeline Summary
# ------------------------------------------------------------------------------

pipeline_summary = pd.DataFrame({

    "Stage":[

        "Load CSV",
        "Schema Validation",
        "Missing Value Handling",
        "Categorical Encoding",
        "Feature Assembly",
        "Train/Test Split",
        "Machine Learning",
        "Evaluation"

    ],

    "Status":[

        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed"

    ]

})

pipeline_summary.to_csv(

    f"{OUTPUT}/t6_dashboard1_pipeline_summary.csv",

    index=False

)

print("Pipeline summary exported ✓")

# ------------------------------------------------------------------------------
# Feature Summary
# ------------------------------------------------------------------------------

feature_summary = pd.DataFrame({

    "Metric":[

        "Total Features",
        "Numeric Features",
        "Categorical Features"

    ],

    "Value":[

        len(df.columns),

        len([

            c for c,d in df.dtypes

            if d in ("int","bigint","double","float")

        ]),

        len([

            c for c,d in df.dtypes

            if d=="string"

        ])

    ]

})

feature_summary.to_csv(

    f"{OUTPUT}/t6_dashboard1_feature_summary.csv",

    index=False

)

print("Feature summary exported ✓")

# ==============================================================================
# DASHBOARD 2
# Machine Learning Performance
# ==============================================================================

print("\nPreparing Dashboard 2...")

metrics=pd.read_csv(
    f"{OUTPUT}/task5_metrics.csv"
)

metrics.to_csv(

    f"{OUTPUT}/t6_dashboard2_model_metrics.csv",

    index=False

)

print("Model metrics loaded ✓")

# ------------------------------------------------------------------------------
# SHAP Feature Importance
# ------------------------------------------------------------------------------

shap_df=pd.read_csv(

    f"{OUTPUT}/task5_shap.csv"

)

shap_df.to_csv(

    f"{OUTPUT}/t6_dashboard2_shap.csv",

    index=False

)

print("SHAP importance loaded ✓")

# ------------------------------------------------------------------------------
# Prediction Time Dataset
# ------------------------------------------------------------------------------

prediction=metrics[

    [

        "Model",

        "Prediction Time (s)"

    ]

]

prediction.to_csv(

    f"{OUTPUT}/t6_dashboard2_prediction_time.csv",

    index=False

)

# ------------------------------------------------------------------------------
# ROC Comparison
# ------------------------------------------------------------------------------

roc=metrics[

    [

        "Model",

        "ROC-AUC"

    ]

]

roc.to_csv(

    f"{OUTPUT}/t6_dashboard2_roc.csv",

    index=False

)

print("Dashboard 2 datasets exported ✓")

print()

print("="*80)
print("COMMIT 1 COMPLETED")
print("="*80)

Mounted at /content/drive
TASK 6 : TABLEAU DATA PREPARATION
Rows    : 11,564,178
Columns : 123

Preparing Dashboard 1...
Dashboard 1 datasets exported ✓
Dataset summary exported ✓
Missing values summary exported ✓
Pipeline summary exported ✓
Feature summary exported ✓

Preparing Dashboard 2...
Model metrics loaded ✓
SHAP importance loaded ✓
Dashboard 2 datasets exported ✓

COMMIT 1 COMPLETED


In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# Commit 2: Prepare Tableau Data for Dashboard 3 & Dashboard 4
# feat(task6): export business intelligence and Spark performance datasets
#
# Coursework Requirement:
# Prepare Tableau-ready datasets for advanced visual analytics.
# Dashboard 3 : Mortgage Business Intelligence
# Dashboard 4 : Spark Performance Analytics
# ──────────────────────────────────────────────────────────────────────────────

from pyspark.sql.functions import *
import pandas as pd

print("="*80)
print("COMMIT 2 : DASHBOARD 3 & DASHBOARD 4")
print("="*80)

# =============================================================================
# DASHBOARD 3
# Mortgage Business Intelligence
# =============================================================================

print("\nPreparing Dashboard 3...")

# -----------------------------------------------------------------------------
# Approval Rate by State
# -----------------------------------------------------------------------------

if "state_code" in df.columns:

    state = (
        df.groupBy("state_code")
          .agg(
              count("*").alias("Applications"),
              sum(col("loan_approval")).alias("Approved")
          )
          .withColumn(
              "Approval Rate (%)",
              round(col("Approved")/col("Applications")*100,2)
          )
          .toPandas()
    )

    state.to_csv(
        f"{OUTPUT}/t6_dashboard3_state.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Loan Purpose
# -----------------------------------------------------------------------------

if "loan_purpose" in df.columns:

    purpose = (
        df.groupBy(
            "loan_purpose",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    purpose.to_csv(
        f"{OUTPUT}/t6_dashboard3_loanpurpose.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Income Bands
# -----------------------------------------------------------------------------

if "income_num" in df.columns:

    income = (
        df.withColumn(
            "Income Band",
            when(col("income_num") < 50, "<50K")
            .when(col("income_num") < 100, "50K-100K")
            .when(col("income_num") < 150, "100K-150K")
            .otherwise(">150K")
        )
        .groupBy(
            "Income Band",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    income.to_csv(
        f"{OUTPUT}/t6_dashboard3_income.csv",
        index=False
    )

    print("Income dashboard exported ✓")

# -----------------------------------------------------------------------------
# Loan Amount Distribution
# -----------------------------------------------------------------------------

if "loan_amount_num" in df.columns:

    loan = (
        df.select("loan_amount_num")
          .toPandas()
    )

    loan.to_csv(
        f"{OUTPUT}/t6_dashboard3_loanamount.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Interest Rate Distribution
# -----------------------------------------------------------------------------

if "interest_rate_num" in df.columns:

    interest = (
        df.select("interest_rate_num")
          .toPandas()
    )

    interest.to_csv(
        f"{OUTPUT}/t6_dashboard3_interest.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Race × Approval
# -----------------------------------------------------------------------------

if "applicant_race_1" in df.columns:

    race = (
        df.groupBy(
            "applicant_race_1",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    race.to_csv(
        f"{OUTPUT}/t6_dashboard3_raceapproval.csv",
        index=False
    )

# -----------------------------------------------------------------------------
# Ethnicity × Approval
# -----------------------------------------------------------------------------

if "applicant_ethnicity_1" in df.columns:

    ethnicity = (
        df.groupBy(
            "applicant_ethnicity_1",
            "loan_approval"
        )
        .count()
        .toPandas()
    )

    ethnicity.to_csv(
        f"{OUTPUT}/t6_dashboard3_ethnicityapproval.csv",
        index=False
    )

print("Dashboard 3 datasets exported ✓")

# =============================================================================
# DASHBOARD 4
# Spark Performance Analytics
# =============================================================================

print("\nPreparing Dashboard 4...")

# -----------------------------------------------------------------------------
# Model Runtime
# -----------------------------------------------------------------------------

metrics = pd.read_csv(
    f"{OUTPUT}/task5_metrics.csv"
)

runtime = metrics[[
    "Model",
    "Prediction Time (s)"
]]

runtime.to_csv(
    f"{OUTPUT}/t6_dashboard4_runtime.csv",
    index=False
)

# -----------------------------------------------------------------------------
# Spark Configuration
# -----------------------------------------------------------------------------

config = pd.DataFrame({

    "Configuration":[
        "Driver Memory",
        "Shuffle Partitions",
        "Spark Version"
    ],

    "Value":[
        spark.conf.get("spark.driver.memory"),
        spark.conf.get("spark.sql.shuffle.partitions"),
        spark.version
    ]

})

config.to_csv(
    f"{OUTPUT}/t6_dashboard4_config.csv",
    index=False
)

# -----------------------------------------------------------------------------
# Cache Benchmark
# -----------------------------------------------------------------------------

cache = pd.DataFrame({

    "Execution":[
        "Cold",
        "Warm"
    ],

    "Time (Seconds)":[
        7.03,
        30.77
    ]

})

cache.to_csv(
    f"{OUTPUT}/t6_dashboard4_cache.csv",
    index=False
)

# -----------------------------------------------------------------------------
# Partition Statistics
# -----------------------------------------------------------------------------

partition = pd.DataFrame({

    "Metric":[
        "Original Partitions",
        "Repartitioned Dataset",
        "Final Partitions",
        "Skew Ratio"
    ],

    "Value":[
        200,
        200,
        128,
        1.92
    ]

})

partition.to_csv(
    f"{OUTPUT}/t6_dashboard4_partition.csv",
    index=False
)

print("Dashboard 4 datasets exported ✓")

print()

print("="*80)
print("COMMIT 2 COMPLETED")
print("="*80)

COMMIT 2 : DASHBOARD 3 & DASHBOARD 4

Preparing Dashboard 3...
Income dashboard exported ✓
Dashboard 3 datasets exported ✓

Preparing Dashboard 4...
Dashboard 4 datasets exported ✓

COMMIT 2 COMPLETED


In [6]:
# ──────────────────────────────────────────────────────────────────────────────
# Task 6 Commit 3: Tableau Public Publication Summary
# feat(task6): document Tableau dashboards and publication details
#
# Coursework Requirement:
# Summarise generated dashboards and prepare Tableau Public publication.
# ──────────────────────────────────────────────────────────────────────────────

import os

print("="*80)
print("COMMIT 3 : TABLEAU PUBLICATION SUMMARY")
print("="*80)

# ==============================================================================
# Generated Tableau Datasets
# ==============================================================================

print("\nGenerated Tableau Datasets\n")

csv_files = sorted([
    f for f in os.listdir(OUTPUT)
    if f.startswith("t6_") and f.endswith(".csv")
])

for file in csv_files:
    print(f"✓ {file}")

print(f"\nTotal Tableau CSV Files : {len(csv_files)}")

# ==============================================================================
# Dashboard Summary
# ==============================================================================

print("\n")
print("="*80)
print("TABLEAU DASHBOARD SUMMARY")
print("="*80)

dashboards = {

    "Dashboard 1":
        "HMDA Mortgage Application Overview",

    "Dashboard 2":
        "Machine Learning Model Performance Dashboard",

    "Dashboard 3":
        "Business Insights Dashboard",

    "Dashboard 4":
        "Scalabilty and Cost Analysis Dashboard"

}

for dashboard, description in dashboards.items():

    print(f"\n{dashboard}")

    print(f"  {description}")

# ==============================================================================
# Dashboard Contents
# ==============================================================================

print("\n")
print("="*80)
print("VISUALISATIONS")
print("="*80)

visuals = {

"Dashboard 1":[
"KPI Summary",
"Loan Approval Distribution",
"Missing Values",
"Loan Purpose Distribution",
"Race Distribution",
"Ethnicity Distribution"
],

"Dashboard 2":[
"Accuracy Comparison",
"Precision Comparison",
"Recall Comparison",
"F1-Score Comparison",
"ROC-AUC Comparison",
"Prediction Time",
"SHAP Feature Importance"
],

"Dashboard 3":[
"Approval Rate by State",
"Approval by Income Band",
"Loan Purpose Analysis",
"Loan Amount Distribution",
"Interest Rate Distribution",
"Race vs Approval",
"Ethnicity vs Approval"
],

"Dashboard 4":[
"Spark Configuration",
"Prediction Runtime",
"Cache Benchmark",
"Partition Analysis",
"Distributed Processing Summary"
]

}

for dashboard, charts in visuals.items():

    print(f"\n{dashboard}")

    for chart in charts:

        print(f"   • {chart}")

# ==============================================================================
# Tableau Public
# ==============================================================================

print("\n")
print("="*80)
print("TABLEAU PUBLIC")
print("="*80)

tableau_url = "https://public.tableau.com/app/profile/bharath.vaderahalli.lokesha/vizzes"

print("\nCurrent URL : https://public.tableau.com/app/profile/bharath.vaderahalli.lokesha/vizzes")

print(tableau_url)

# ==============================================================================
# Final Summary
# ==============================================================================

print("\n")
print("="*80)
print("TASK 6 COMPLETED SUCCESSFULLY")
print("="*80)

print("""
Four interactive Tableau dashboards were prepared using
datasets generated throughout the HMDA mortgage approval
analytics workflow.

The dashboards provide:

• Mortgage application overview

• Machine learning model evaluation

• Business intelligence insights

• Spark performance and scalability analytics

These dashboards are ready for publication through Tableau Public.
""")

COMMIT 3 : TABLEAU PUBLICATION SUMMARY

Generated Tableau Datasets

✓ t6_dashboard1_approval.csv
✓ t6_dashboard1_dataset_summary.csv
✓ t6_dashboard1_ethnicity.csv
✓ t6_dashboard1_feature_summary.csv
✓ t6_dashboard1_kpi.csv
✓ t6_dashboard1_loanpurpose.csv
✓ t6_dashboard1_missing.csv
✓ t6_dashboard1_missing_values.csv
✓ t6_dashboard1_pipeline_summary.csv
✓ t6_dashboard1_race.csv
✓ t6_dashboard2_model_metrics.csv
✓ t6_dashboard2_prediction_time.csv
✓ t6_dashboard2_roc.csv
✓ t6_dashboard2_shap.csv
✓ t6_dashboard3_ethnicityapproval.csv
✓ t6_dashboard3_income.csv
✓ t6_dashboard3_interest.csv
✓ t6_dashboard3_loanamount.csv
✓ t6_dashboard3_loanpurpose.csv
✓ t6_dashboard3_raceapproval.csv
✓ t6_dashboard3_state.csv
✓ t6_dashboard4_cache.csv
✓ t6_dashboard4_config.csv
✓ t6_dashboard4_partition.csv
✓ t6_dashboard4_runtime.csv

Total Tableau CSV Files : 25


TABLEAU DASHBOARD SUMMARY

Dashboard 1
  HMDA Mortgage Application Overview

Dashboard 2
  Machine Learning Model Performance Dashboard

Dashb